In [0]:
%python
# M1 - Bronze Ingestion - Planner Overrides
RAW = "/Volumes/workspace/rp_landing/raw"
OPS = "/Volumes/workspace/rp_landing/ops"

from pyspark.sql import functions as F

(spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", f"{OPS}/schemas/planner_overrides")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .load(f"{RAW}/landing/planner_overrides")
  .select("*", F.col("_metadata.file_path").alias("_source_file"),
          F.current_timestamp().alias("_ingest_ts"))
  .writeStream
  .option("checkpointLocation", f"{OPS}/chk/bronze_planner_overrides")
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .toTable("workspace.rp_bronze.planner_overrides")
)

In [0]:
SELECT count(*) FROM workspace.rp_bronze.planner_overrides